# Specialty Coffee Sales — Time Series Forecasting

**Assignments 1–3 | Original reproducible notebook**

**Business problem:** Forecast monthly units sold by a fictional specialty-coffee retailer using synthetic data spanning **January 2018–December 2025**. The values were newly generated for this exercise (random seed `314159`); they are **not actual commercial sales**.

**Objectives**
1. Separate sales into trend, seasonal, and irregular components with STL decomposition.
2. Test stationarity with Augmented Dickey–Fuller (ADF) and KPSS; evaluate transformations.
3. Train an ARIMA model and a Prophet model; assess the final 12 months using MAE, RMSE, and MAPE, plus a seasonal-naïve reference.

**Methodological decisions:** Monthly frequency (`MS`), additive STL with period 12, an untouched 2025 holdout, and model selection using **training data only**. Promotions are shown descriptively but **not** used as model features, to keep comparison fair.

> Reproduce in Colab, JupyterLab, or VS Code. Install dependencies from `requirements.txt` first. In Colab, run `%pip install -q -r requirements.txt` after uploading the files or using a GitHub checkout.

**Automated execution:** `.github/workflows/evaluate.yml` installs Prophet and runs all notebook cells, then commits the generated evaluation CSVs and notebook outputs to GitHub. The CSV files committed before this workflow runs are **not** final Prophet results.


**Quick start:** This notebook is self-contained and loads data without a separate CSV file. To evaluate Prophet, install it in the runtime (`%pip install prophet`) before running all cells.

## Setup and data integrity

**Portable notebook:** the exact 96-row dataset is embedded as a fallback. You can upload **just the `.ipynb` file** to Colab; a separate CSV is optional. If the CSV exists beside the notebook, it is used instead.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': 0.25})
SEED = 314159
# Self-contained dataset: use the CSV when available; otherwise load the
# exact same embedded synthetic observations. Works in Jupyter and Colab
# even when only the .ipynb file was uploaded.
import base64
import io
import zlib
DATA_PATH = Path('coffee_sales_monthly.csv')
EMBEDDED_CSV_ZLIB_BASE64 = (
    'eNpNlUmOGzEMRfc5iwNQnHmaRhYB0kBsB3Hn/qkqDjJgePHMIv9TSfL9+fj6dXv9+P3z9fHv8fn1uv35+7w/vz6fj4/7+ds3hOXf'
    'YR2f2wI5vqEQXgg5YBAl0iWD+EIMHIPkQmoRt1VIq0p8qqx64Z7oF1pGu33kg6zTfsGFyBQHZXqL87tQpjdXzqpox6WMg8rR1xqU'
    'jsRkg9JRoHNFOzqI5sRoRzHwqUpHcYpB6ajrLUQ5+hpUjmo06cvR6QhRE9MRQTHfEEI7koMOSkchkEHpKEi7iiuX7V7paMjpeKJ0'
    'VGWeqnRUpzWoHF03inodNg+WowHRoCs9HimwJ5bjseUq12pHBV+DytF5I6r1Qh/EtU1s95KceKxYTlzt6Kg6VeloxjIoHY1xT4xa'
    'HLdG6YhAvFE6ooT1xHaE2oWI44h1YE6EvaobUbaHoEHpGLZskPTEPB0nSsdAhamy6mU7RDnSG4qqImxUjkQhg9KRALAnlqOQVFRq'
    'R0OVQenoUmfoRFR7om6mE3GG8LeqdDzuCciJVI5IETxVVr0EB3kipl2VjkezydWODpO+HE3Ne2I5mnnl4nYM68XhdozZ99yOGJ2L'
    '25G0c/E4hlJO5DfHPTEdyTAGeS1Onw5uRz6uq0bleNxMOigdw8F6YjrSsXWql5QjrvO+b1Q7ui++E6WjrNaWdhTvXSjt6AvKUdpR'
    '5paTdpT1NjEdVWy3j3odax5sR68/sBOtFDqPaE0sR5Jr5/wH8B9cPw=='
)
if DATA_PATH.exists():
    raw = pd.read_csv(DATA_PATH, parse_dates=['month'])
    print(f'Loaded data from {DATA_PATH.resolve()}')
else:
    csv_bytes = zlib.decompress(base64.b64decode(EMBEDDED_CSV_ZLIB_BASE64))
    raw = pd.read_csv(io.BytesIO(csv_bytes), parse_dates=['month'])
    print('CSV file not found; loaded the identical embedded sales dataset.')
assert raw['month'].is_unique and raw['sales_units'].notna().all()
assert raw['sales_units'].gt(0).all()
series = raw.set_index('month')['sales_units'].asfreq('MS').astype(float)
assert not series.isna().any(), 'Monthly observations are missing.'
print(f'Months: {len(series)} | {series.index.min():%Y-%m} to {series.index.max():%Y-%m}')
display(raw.head())
display(series.describe().to_frame('sales_units').round(2))

In [ ]:
fig, ax = plt.subplots()
series.plot(ax=ax, linewidth=1.7, label='Monthly sales')
ax.set(title='Monthly coffee sales (synthetic)', xlabel='', ylabel='Units sold')
ax.legend(); plt.tight_layout(); plt.show()
print('Top 3 months by sales:')
display(series.nlargest(3).rename('units').to_frame())

## Assignment 1 — Trend, seasonality, residuals

Use **STL** (Seasonal-Trend decomposition using LOESS). STL is more adaptable to slowly evolving patterns than a fixed classical decomposition. With an **additive** decomposition:

\[y_t = T_t + S_t + R_t\]

Here `T` is smoothed trend, `S` is a recurring 12-month seasonal effect, and `R` is the remaining irregular fluctuation. Decomposition is exploratory: plotting the full series is **not** used to choose model parameters, thereby avoiding holdout-based tuning.

In [ ]:
stl = STL(series, period=12, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(11, 8)
fig.suptitle('STL decomposition | Monthly coffee sales', y=1.01)
plt.tight_layout(); plt.show()
check = np.max(np.abs(series.values-(stl.trend+stl.seasonal+stl.resid)))
print(f'Maximum numerical reconstruction error: {check:.9f}')
monthly_effect = pd.DataFrame({'month':series.index.month, 'seasonal_component':stl.seasonal.values}).groupby('month')['seasonal_component'].mean()
print('Average seasonal contribution by calendar month (units):')
display(monthly_effect.round(0).to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(10,3))
monthly_effect.plot.bar(ax=ax, color='#287c91')
ax.set(title='Average within-year STL seasonal effect', xlabel='Month (1=Jan)', ylabel='Units versus trend')
plt.tight_layout(); plt.show()
print('Interpretation: positive bars indicate typically stronger months, while the residual captures irregular shocks.')

## Assignment 2 — ADF, KPSS, and transformations

The two tests have **opposite null hypotheses**:

- **ADF:** null = a unit root exists (non-stationary). A small p-value suggests rejecting non-stationarity.
- **KPSS:** null = stationary around a level (with `regression="c"`). A small p-value suggests rejecting stationarity.

Use **α = 0.05**. The tests may disagree, particularly when deterministic trend/seasonal structures exist. They are diagnostics, not absolute proof of stationarity.

Transformations checked: raw sales, first difference, seasonal difference (lag 12), and first + seasonal difference. The latter may remove both trend and annual seasonality, though unnecessary differencing can increase noise.

In [ ]:
def stationarity_tests(x, name):
    x = pd.Series(x).dropna().astype(float)
    adf = adfuller(x, autolag='AIC')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        kp = kpss(x, regression='c', nlags='auto')
    return {'series': name, 'observations': len(x), 'ADF statistic': adf[0],
            'ADF p-value': adf[1], 'KPSS statistic': kp[0], 'KPSS p-value': kp[1],
            'ADF rejects unit root': adf[1] < 0.05,
            'KPSS retains stationarity': kp[1] >= 0.05}

transformations = {
    'Raw sales': series,
    'First difference (lag 1)': series.diff(),
    'Seasonal difference (lag 12)': series.diff(12),
    'First + seasonal difference': series.diff().diff(12),
}
results = pd.DataFrame([stationarity_tests(v,k) for k,v in transformations.items()]).set_index('series')
display(results.round(4))
print('Practical rule: stronger support for stationarity means ADF p < 0.05 and KPSS p >= 0.05.')

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(11, 10))
for ax, (name, values) in zip(axs, transformations.items()):
    values.plot(ax=ax, color='#236d8c', linewidth=1.25)
    ax.set_title(name); ax.set_xlabel('')
plt.tight_layout(); plt.show()

**Interpretation:** Inspect the generated test table rather than assuming one transformation must pass both tests. Seasonal differencing removes much of a repeating annual pattern; first differencing can remove deterministic or stochastic drift. **Do not fit differencing choices based on the test period**—the ARIMA search below uses only the training split.

## Assignment 3 — Out-of-sample forecasting and comparison

Reserve **January–December 2025** for a final, untouched test period (12 consecutive months). Fit on 2018–2024. Select ARIMA `(p,d,q)` using training-set AIC only. Prophet gets yearly seasonality and a **linear-growth** trend (default), with weekly/daily seasonality disabled because the observations are monthly. Evaluate both on exactly the same future dates.

In [ ]:
train = series.loc[:'2024-12-01']
test = series.loc['2025-01-01':]
assert len(test) == 12 and train.index.max() < test.index.min()
print(f'Train: {train.index.min():%Y-%m}–{train.index.max():%Y-%m}, n={len(train)}')
print(f'Test:  {test.index.min():%Y-%m}–{test.index.max():%Y-%m}, n={len(test)}')
fig, ax=plt.subplots()
train.plot(ax=ax,label='Train'); test.plot(ax=ax,label='Test',color='orange')
ax.set(title='Chronological split; no random shuffling',xlabel='',ylabel='Units');ax.legend();plt.tight_layout();plt.show()

In [ ]:
# Modest grid to keep execution time practical; d=1 follows strong trend in the series.
# Crucially, ONLY train is used to choose the order.
search_rows = []
for p in (0,1,2):
    for q in (0,1,2):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                candidate = ARIMA(train, order=(p,1,q), trend='n').fit()
            search_rows.append({'order': (p,1,q), 'AIC': candidate.aic})
        except (ValueError, np.linalg.LinAlgError) as exc:
            print(f'Skipping {(p,1,q)}: {exc}')
if not search_rows:
    raise RuntimeError('ARIMA grid search failed for all combinations.')
aic_table = pd.DataFrame(search_rows).sort_values('AIC').reset_index(drop=True)
display(aic_table)
best_order = tuple(aic_table.loc[0,'order'])
print('Selected ARIMA order:', best_order)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    arima_fit=ARIMA(train,order=best_order,trend='n').fit()
arima_prediction=arima_fit.forecast(steps=len(test))
arima_prediction.index=test.index
print(arima_fit.summary().tables[0])

### Prophet — actual model training and evaluation

The project requires the real `prophet.Prophet` package. Run `pip install -r requirements.txt` first or use the GitHub Actions workflow bundled with the repository; the workflow installs dependencies, executes this entire notebook, and commits the **actual Prophet forecasts and metrics**. Forecasts are made exclusively for the unseen 2025 months.

No surrogate is substituted for Prophet. This notebook raises a clear installation error instead of quietly presenting an incomplete comparison as final.

In [ ]:
# Evaluate the actual Prophet model; mandatory for a complete comparison.
try:
    from prophet import Prophet
except ImportError as exc:
    raise ImportError(
        'The actual Prophet package is required. Run: pip install -r requirements.txt '
        'or push this project to GitHub and run the included GitHub Actions workflow.'
    ) from exc

prophet_train = train.rename_axis('ds').reset_index(name='y')
prophet_model = Prophet(
    growth='linear', yearly_seasonality=True,
    weekly_seasonality=False, daily_seasonality=False,
    seasonality_mode='additive', changepoint_prior_scale=0.08,
    interval_width=0.95, uncertainty_samples=0,
)
prophet_model.fit(prophet_train)
prophet_forecast = prophet_model.predict(pd.DataFrame({'ds': test.index}))
prophet_prediction = pd.Series(
    prophet_forecast['yhat'].to_numpy(), index=test.index, name='Prophet'
)
assert prophet_prediction.notna().all() and len(prophet_prediction) == len(test)
print('REAL Prophet fit complete — forecasting 12 held-out months in 2025.')
display(prophet_forecast[['ds', 'yhat']].round(1))

In [ ]:
# Comparison uses exactly the same 12-month period for all models.
seasonal_naive = pd.Series(train.iloc[-12:].to_numpy(), index=test.index, name='Seasonal naive')
predictions = pd.DataFrame({
    'Actual': test, 'ARIMA': arima_prediction,
    'Prophet': prophet_prediction, 'Seasonal naive': seasonal_naive,
})
assert not predictions.isna().any().any()
predictions.rename_axis('month').to_csv('forecast_predictions_2025.csv')
display(predictions.round(0))
fig, ax = plt.subplots(figsize=(12, 5))
for column in predictions.columns:
    ax.plot(predictions.index, predictions[column], marker='o', label=column)
ax.set(title='2025 test-set forecasts — equal holdout', xlabel='Month', ylabel='Units sold')
ax.legend(ncol=4); plt.tight_layout(); plt.show()

In [ ]:
def forecast_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if np.any(actual == 0):
        raise ValueError('MAPE is undefined when actual value is zero.')
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'MAPE (%)': np.mean(np.abs((actual - predicted) / actual)) * 100,
    }
comparison = pd.DataFrame({
    name: forecast_metrics(test, predictions[name])
    for name in ('ARIMA', 'Prophet', 'Seasonal naive')
}).T.sort_values('MAE')
comparison.index.name = 'Model'
comparison.to_csv('forecast_metrics_2025.csv')
display(comparison.round(2))
print('Lowest test MAE:', comparison.index[0])
assert 'Prophet' in comparison.index and np.isfinite(comparison.loc['Prophet']).all()
print('All three models evaluated successfully, including REAL Prophet.')

In [ ]:
fig, axs=plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axs, ['MAE', 'RMSE', 'MAPE (%)']):
    comparison[metric].plot.bar(ax=ax)
    ax.set_title(metric); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Lower is better — 2025 chronological holdout', y=1.03)
plt.tight_layout(); plt.show()
predictions.rename_axis('month').to_csv('forecast_predictions_2025.csv')
comparison.rename_axis('model').to_csv('forecast_metrics_2025.csv')
print('Saved forecast_predictions_2025.csv and forecast_metrics_2025.csv')
print('Full ARIMA vs Prophet comparison complete:', prophet_prediction is not None)

## Findings and limitations

**How to report the result:** Read the computed table above to identify the model with the lowest held-out MAE, RMSE, and MAPE. **Do not assume Prophet wins**; the winner depends on actual prediction error. If the metrics disagree, state which business loss matters most (absolute error or large misses).

**Interpretation checklist:**
- Increasing smoothed STL trend indicates growth; within-year STL swings indicate repeatable yearly variation.
- ADF low p-value and KPSS high p-value together support level-stationarity; discuss disagreements explicitly.
- ARIMA captures autocorrelation and trend through differencing; unseasonal ARIMA can struggle with yearly variation.
- Prophet has an explicit annual seasonal component and trend changepoints, which may be advantageous on this dataset.
- Seasonal-naïve is a useful reality check: a complex model should add value beyond simply repeating last year.

**Limitations:** Synthetic data; single product family; no price, inventory, holidays, external drivers, or uncertainty calibration. Only one 12-month holdout is evaluated. Stronger production evaluation would use rolling-origin backtesting, drift monitoring, forecast intervals, and the costs of over/under-forecasting. No modeling claims about real stores are justified.

**Reproducibility:** Fixed synthetic dataset supplied in CSV; fixed train/test periods; deterministic ARIMA selection; Prophet is actually fitted through `prophet.Prophet`, with dependency installation ensured by the included GitHub Actions workflow. This is a forecasting **analysis notebook**, not a deployed prediction API.